# Homework 1 — Problem 1: Modified AlexNet on CIFAR-10

**Author:** *(your name)*  
**Student ID:** *(your ID)*  
**GitHub:** *(your repo link)*  
**Seed:** 42 | **Batch size:** 128 | **Device:** auto-detected below

---

### Notebook structure
| Cell | Contents |
|------|----------|
| 1 | Environment & hardware check |
| 2 | Install / verify dependencies |
| 3 | Data setup — CIFAR-10 loaders |
| 4 | Part A — Modified AlexNet architecture + training |
| 5 | Part A — Evaluation, plots, filter visualization |
| 6 | Part B — Dropout variants (p=0.3, p=0.5) training |
| 7 | Part B — Comparison plots & summary table |


## Cell 1 — Environment & Hardware Check
Run this first. It detects your GPU and configures the notebook accordingly.

In [1]:
import sys, platform
import torch

print("=" * 55)
print("  SYSTEM INFORMATION")
print("=" * 55)
print(f"  OS            : {platform.system()} {platform.release()}")
print(f"  Python        : {sys.version.split()[0]}")
print(f"  PyTorch       : {torch.__version__}")
print()

# GPU detection
cuda_available = torch.cuda.is_available()
print(f"  CUDA available: {cuda_available}")
print(f"  CUDA version  : {torch.version.cuda}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU           : {gpu_name}")
    print(f"  GPU memory    : {mem_total:.1f} GB")
    DEVICE = torch.device("cuda")
    print()
    print("  ✅ GPU detected — training will be fast.")
    print("     If you expected a GPU and see CPU below, see the")
    print("     reinstall instructions in the next cell.")
else:
    DEVICE = torch.device("cpu")
    import os
    cpu_cores = os.cpu_count()
    print(f"  CPU cores     : {cpu_cores}")
    print()
    print("  ⚠️  No GPU detected — running on CPU.")
    print("     Training will be slower (~30-60 min for 30 epochs).")
    print("     See Cell 2 for GPU reinstall instructions.")

print()
print(f"  Active device : {DEVICE}")
print("=" * 55)


  SYSTEM INFORMATION
  OS            : Windows 11
  Python        : 3.12.8
  PyTorch       : 2.6.0+cu124

  CUDA available: True
  CUDA version  : 12.4
  GPU           : NVIDIA GeForce MX250
  GPU memory    : 2.1 GB

  ✅ GPU detected — training will be fast.
     If you expected a GPU and see CPU below, see the
     reinstall instructions in the next cell.

  Active device : cuda


## Cell 2 — Dependencies

### If Cell 1 showed CPU but you have an NVIDIA GPU:
Your PyTorch was installed without CUDA support. Reinstall it:
```
pip uninstall torch torchvision -y
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
```
Then **restart the kernel** and re-run from Cell 1.

### Install all required packages (run once):

In [2]:
# Run this cell once to install all dependencies.
# If everything is already installed it will just confirm versions.
import subprocess, sys

packages = ["torchvision", "torchinfo", "scikit-learn", "matplotlib", "numpy"]
for pkg in packages:
    try:
        __import__(pkg.replace("-","_").split("[")[0])
        print(f"  ✅ {pkg} already installed")
    except ImportError:
        print(f"  📦 Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  ✅ {pkg} installed")

# Verify torchinfo specifically (commonly missing)
try:
    from torchinfo import summary
    print("  ✅ torchinfo ready")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchinfo", "-q"])
    from torchinfo import summary
    print("  ✅ torchinfo installed and ready")

print()
print("All dependencies satisfied.")


  ✅ torchvision already installed
  ✅ torchinfo already installed
  📦 Installing scikit-learn...
  ✅ scikit-learn installed
  ✅ matplotlib already installed
  ✅ numpy already installed
  ✅ torchinfo ready

All dependencies satisfied.


## Cell 3 — Data Setup
Defines the shared CIFAR-10 data pipeline used across all three problems. Run once — subsequent cells import from this.

In [3]:
import os
import torch
import numpy as np
import warnings
warnings.filterwarnings("ignore")   # suppress the pin_memory + numpy deprecation warnings
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# ── Global config ─────────────────────────────────────────────────────────────
RANDOM_SEED  = 42
BATCH_SIZE   = 128
NUM_WORKERS  = 0          # 0 is safest on Windows in notebooks
VAL_FRACTION = 0.1
DATA_DIR     = "./data"

# ── CIFAR-10 normalization stats ───────────────────────────────────────────────
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# ── Transforms ────────────────────────────────────────────────────────────────
# Training: random crop + horizontal flip, then normalize.
# Val/Test: normalize only — no augmentation.
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# ── DataLoader builder ────────────────────────────────────────────────────────
def get_cifar10_loaders(
    data_dir=DATA_DIR, batch_size=BATCH_SIZE,
    val_fraction=VAL_FRACTION, seed=RANDOM_SEED,
    num_workers=NUM_WORKERS,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Load train set twice: augmented for training, clean for validation
    full_train_aug  = datasets.CIFAR10(data_dir, train=True,  download=True,  transform=train_transform)
    full_train_eval = datasets.CIFAR10(data_dir, train=True,  download=False, transform=eval_transform)
    test_dataset    = datasets.CIFAR10(data_dir, train=False, download=False, transform=eval_transform)

    n_total = len(full_train_aug)
    n_val   = int(n_total * val_fraction)
    n_train = n_total - n_val

    rng     = np.random.default_rng(seed)
    indices = rng.permutation(n_total)
    train_indices, val_indices = indices[:n_train], indices[n_train:]

    train_loader = DataLoader(Subset(full_train_aug,  train_indices),
                              batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, drop_last=True)
    val_loader   = DataLoader(Subset(full_train_eval, val_indices),
                              batch_size=batch_size, shuffle=False,
                              num_workers=num_workers)
    test_loader  = DataLoader(test_dataset,
                              batch_size=batch_size, shuffle=False,
                              num_workers=num_workers)
    return train_loader, val_loader, test_loader

# ── Sanity check ──────────────────────────────────────────────────────────────
train_loader, val_loader, test_loader = get_cifar10_loaders()

print(f"Seed          : {RANDOM_SEED}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Val fraction  : {VAL_FRACTION}")
print()
print(f"Train batches : {len(train_loader)}  ({len(train_loader.dataset):,} samples)")
print(f"Val   batches : {len(val_loader)}  ({len(val_loader.dataset):,} samples)")
print(f"Test  batches : {len(test_loader)}  ({len(test_loader.dataset):,} samples)")

images, labels = next(iter(train_loader))
print()
print(f"Batch shape   : {images.shape}")
print(f"Pixel range   : {images.min():.3f} / {images.max():.3f}  (normalized)")
print(f"Sample labels : {[CIFAR10_CLASSES[l] for l in labels[:8].tolist()]}")
print()
print("✅ Data setup looks good.")


Seed          : 42
Batch size    : 128
Val fraction  : 0.1

Train batches : 351  (45,000 samples)
Val   batches : 40  (5,000 samples)
Test  batches : 79  (10,000 samples)

Batch shape   : torch.Size([128, 3, 32, 32])
Pixel range   : -1.989 / 2.126  (normalized)
Sample labels : ['cat', 'deer', 'airplane', 'frog', 'frog', 'cat', 'dog', 'bird']

✅ Data setup looks good.


## Cell 4 — Part A: Modified AlexNet Architecture & Training

**Architecture changes from original AlexNet (justified):**

| Layer | Original | Modified | Reason |
|-------|----------|----------|--------|
| Conv1 | 11×11 stride 4 | 3×3 stride 1 | 11×11 s4 on 32×32 collapses to 6×6 immediately |
| Pool after Conv1 | 3×3 s2 | **Removed** | No spatial headroom left at this resolution |
| Conv2 | 5×5, 256 filters | 3×3, 192 filters | Comparable receptive field, fewer params |
| Conv3/4/5 | 384, 384, 256 | 192, 192, 128 | Halved — proportional to smaller input |
| FC layers | 4096 → 4096 → 1000 | 1024 → 1024 → 10 | 4096 sized for ImageNet; 1024 sufficient for 10 classes |
| LRN | Present | **Removed** | No benefit at CIFAR-10 scale |


In [4]:
import os, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from torchinfo import summary as torchinfo_summary
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# ── Device verification ───────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 55)
print("  DEVICE CHECK")
print("=" * 55)
print(f"  CUDA available : {torch.cuda.is_available()}")
print(f"  Active device  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU name       : {torch.cuda.get_device_name(0)}")
    print(f"  GPU memory     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("  ✅ Running on GPU")
else:
    print("  ⚠️  Running on CPU — check kernel selection (top right corner)")
    print("  Expected: 'Python 3.12 (deep learning)'")
print("=" * 55)
print()

assert DEVICE.type == "cuda", "❌ Not on GPU! Switch kernel to 'Python 3.12 (deep learning)' and restart."

# Force CUDA to initialize before training
torch.cuda.empty_cache()
_ = torch.zeros(1).to(DEVICE)
print(f"CUDA initialized: {torch.cuda.memory_allocated(DEVICE) >= 0}")

# ── Hyperparameters ───────────────────────────────────────────────────────────
SEED         = 42
NUM_EPOCHS   = 30
LR           = 0.001
MOMENTUM     = 0.9
WEIGHT_DECAY = 5e-4
LR_STEP_SIZE = 10
LR_GAMMA     = 0.1
NUM_CLASSES  = 10
SAVE_DIR_P1A = "./outputs_p1a"
os.makedirs(SAVE_DIR_P1A, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# ── Architecture ──────────────────────────────────────────────────────────────
class ModifiedAlexNet(nn.Module):
    """
    Spatial flow:
      Input        : 3  x 32 x 32
      After conv1  : 64 x 32 x 32
      After conv2  : 192x 32 x 32
      After pool1  : 192x 16 x 16
      After conv3  : 192x 16 x 16
      After conv4  : 192x 16 x 16
      After conv5  : 128x 16 x 16
      After pool2  : 128x  8 x  8
      Flatten      : 8192
      FC1 -> FC2 -> FC3(10)
    """
    def __init__(self, num_classes=NUM_CLASSES, dropout_p=0.0):
        super().__init__()
        self.dropout_p = dropout_p
        self.features = nn.Sequential(
            nn.Conv2d(3,   64,  3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(64,  192, 3, 1, 1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(192, 192, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(192, 192, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(192, 128, 3, 1, 1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(128 * 8 * 8, 1024), nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(1024, 1024),         nn.ReLU(inplace=True),
            nn.Linear(1024, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        # Kaiming (He) initialization — designed for ReLU networks.
        # Original AlexNet used std=0.01 which works for their large network
        # with LRN layers, but causes dead neurons on our smaller model.
        # Kaiming scales weights by sqrt(2/fan_in) ensuring gradients flow.
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# ── Training utilities ────────────────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += out.max(1)[1].eq(labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        out  = model(images)
        loss = criterion(out, labels)
        total_loss += loss.item() * images.size(0)
        correct    += out.max(1)[1].eq(labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def get_all_predictions(model, loader, device):
    model.eval()
    all_labels, all_preds = [], []
    for images, labels in loader:
        out = model(images.to(device))
        all_preds.extend(out.max(1)[1].cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

def run_training(model, train_loader, val_loader,
                 num_epochs=NUM_EPOCHS, lr=LR,
                 label="model", save_dir=SAVE_DIR_P1A):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr,
                          momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(
        optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

    history = dict(train_loss=[], val_loss=[],
                   train_acc=[], val_acc=[], epoch_times=[])
    best_val_acc = 0.0
    ckpt_path    = os.path.join(save_dir, f"{label}_best.pt")

    print(f"\nTraining : {label}")
    print(f"Device   : {DEVICE}  |  Epochs: {num_epochs}  |  LR: {lr}  |  BS: {BATCH_SIZE}")
    print(f"{'─'*75}")

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        vl, va = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()
        elapsed = time.time() - t0

        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["train_acc"].append(ta)
        history["val_acc"].append(va)
        history["epoch_times"].append(elapsed)

        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), ckpt_path)

        print(f"Epoch [{epoch:3d}/{num_epochs}]  "
              f"Train Loss: {tl:.4f}  Acc: {ta*100:.2f}%  |  "
              f"Val Loss: {vl:.4f}  Acc: {va*100:.2f}%  |  "
              f"LR: {scheduler.get_last_lr()[0]:.5f}  ({elapsed:.1f}s)")

    print(f"{'─'*75}")
    print(f"Best val accuracy : {best_val_acc*100:.2f}%  →  saved: {ckpt_path}\n")
    return history, ckpt_path

# ── torchinfo summary ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  Modified AlexNet — Architecture Summary")
print("=" * 60)
set_seed(SEED)
model_p1a = ModifiedAlexNet(num_classes=NUM_CLASSES, dropout_p=0.0)
torchinfo_summary(model_p1a, input_size=(1, 3, 32, 32),
                  col_names=["input_size", "output_size", "num_params"],
                  verbose=1, device=DEVICE)

total_params = sum(p.numel() for p in model_p1a.parameters() if p.requires_grad)
print(f"\nOriginal AlexNet params : ~61,000,000")
print(f"This model params       :  {total_params:,}")
print(f"Reduction factor        : ~{61_000_000 // total_params}x")

# ── Verify initialization before training ────────────────────────────────────
first_conv = next(m for m in model_p1a.modules() if isinstance(m, nn.Conv2d))
print(f"\nInit check — first conv weight std : {first_conv.weight.std().item():.4f}  (expect ~0.15+, not 0.01)")
print(f"Init check — first conv bias mean  : {first_conv.bias.mean().item():.4f}   (expect 0.0)")
print()

# ── Train ─────────────────────────────────────────────────────────────────────
set_seed(SEED)
history_p1a, ckpt_p1a = run_training(
    model_p1a, train_loader, val_loader,
    label="alexnet_baseline_p1a", save_dir=SAVE_DIR_P1A
)
print("✅ Part A training complete.")


  DEVICE CHECK
  CUDA available : True
  Active device  : cuda
  GPU name       : NVIDIA GeForce MX250
  GPU memory     : 2.1 GB
  ✅ Running on GPU

CUDA initialized: True

  Modified AlexNet — Architecture Summary
Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
ModifiedAlexNet                          [1, 3, 32, 32]            [1, 10]                   --
├─Sequential: 1-1                        [1, 3, 32, 32]            [1, 128, 8, 8]            --
│    └─Conv2d: 2-1                       [1, 3, 32, 32]            [1, 64, 32, 32]           1,792
│    └─ReLU: 2-2                         [1, 64, 32, 32]           [1, 64, 32, 32]           --
│    └─Conv2d: 2-3                       [1, 64, 32, 32]           [1, 192, 32, 32]          110,784
│    └─ReLU: 2-4                         [1, 192, 32, 32]          [1, 192, 32, 32]          --
│    └─MaxPool2d: 2-5                    [1, 192, 32, 32]          [1, 192, 16, 16]          --
│   

## Cell 5 — Part A: Evaluation, Plots & Filter Visualization

In [5]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
model_p1a.load_state_dict(torch.load(ckpt_p1a, map_location=DEVICE))
model_p1a.to(DEVICE)
criterion = nn.CrossEntropyLoss()

test_loss_p1a, test_acc_p1a = evaluate(model_p1a, test_loader, criterion, DEVICE)
print(f"Final Test Accuracy : {test_acc_p1a*100:.2f}%")
print(f"Final Test Loss     : {test_loss_p1a:.4f}")

# ── Loss & accuracy curves ────────────────────────────────────────────────────
epochs = range(1, len(history_p1a["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, history_p1a["train_loss"], label="Train Loss")
axes[0].plot(epochs, history_p1a["val_loss"],   label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Part A — Loss Curves"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [a*100 for a in history_p1a["train_acc"]], label="Train Acc")
axes[1].plot(epochs, [a*100 for a in history_p1a["val_acc"]],   label="Val Acc")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Part A — Accuracy Curves"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR_P1A, "alexnet_baseline_p1a_curves.png"), dpi=150)
plt.show()
print("Curves saved.")

# ── Confusion matrix ──────────────────────────────────────────────────────────
true_p1a, pred_p1a = get_all_predictions(model_p1a, test_loader, DEVICE)
cm = confusion_matrix(true_p1a, pred_p1a, normalize="true")
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=CIFAR10_CLASSES).plot(
    ax=ax, cmap="Blues", colorbar=True, xticks_rotation=45)
ax.set_title("Part A — Confusion Matrix (normalized)")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR_P1A, "alexnet_baseline_p1a_confusion.png"), dpi=150)
plt.show()
print("Confusion matrix saved.")

# ── First-layer filter visualization ─────────────────────────────────────────
weights = next(m for m in model_p1a.modules()
               if isinstance(m, nn.Conv2d)).weight.detach().cpu()
n = weights.shape[0]   # 64
fig, axes = plt.subplots(8, 8, figsize=(12, 12))
for idx in range(64):
    ax = axes[idx // 8][idx % 8]
    ax.axis("off")
    f = weights[idx] - weights[idx].min()
    if f.max() > 0: f = f / f.max()
    ax.imshow(f.permute(1, 2, 0).numpy())
fig.suptitle("Part A — First Conv Layer Filters (64 × 3×3×3)", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR_P1A, "alexnet_baseline_p1a_filters.png"), dpi=150)
plt.show()
print("Filter visualization saved.")

# ── Report summary ────────────────────────────────────────────────────────────
avg_time = np.mean(history_p1a["epoch_times"])
print()
print("=" * 55)
print("  REPORT SUMMARY — Problem 1 Part A")
print("=" * 55)
print(f"  Architecture      : Modified AlexNet")
print(f"  Trainable params  : {total_params:,}")
print(f"  Original AlexNet  : ~61,000,000")
print(f"  Epochs trained    : {NUM_EPOCHS}")
print(f"  Best val accuracy : {max(history_p1a['val_acc'])*100:.2f}%")
print(f"  Final test acc    : {test_acc_p1a*100:.2f}%")
print(f"  Avg epoch time    : {avg_time:.1f}s")
print(f"  Device            : {DEVICE}")
print(f"  Seed              : {SEED}")
print(f"  Optimizer         : SGD  lr={LR}  momentum={MOMENTUM}  wd={WEIGHT_DECAY}")
print(f"  Scheduler         : StepLR  step={LR_STEP_SIZE}  gamma={LR_GAMMA}")
print(f"  Batch size        : {BATCH_SIZE}")
print("=" * 55)


Final Test Accuracy : 72.48%
Final Test Loss     : 0.7754
Curves saved.
Confusion matrix saved.
Filter visualization saved.

  REPORT SUMMARY — Problem 1 Part A
  Architecture      : Modified AlexNet
  Trainable params  : 10,447,306
  Original AlexNet  : ~61,000,000
  Epochs trained    : 30
  Best val accuracy : 73.28%
  Final test acc    : 72.48%
  Avg epoch time    : 126.0s
  Device            : cuda
  Seed              : 42
  Optimizer         : SGD  lr=0.001  momentum=0.9  wd=0.0005
  Scheduler         : StepLR  step=10  gamma=0.1
  Batch size        : 128


## Cell 6 — Part B: Dropout Variants Training

Trains three variants under **identical conditions** — only `dropout_p` changes:
- **Baseline** — `p = 0.0` (no dropout)
- **Dropout 0.3** — `p = 0.3`  
- **Dropout 0.5** — `p = 0.5` (recommended by Srivastava et al. as near-optimal)

Dropout is applied before the first two FC layers, matching the original AlexNet paper placement.


In [6]:
SAVE_DIR_P1B  = "./outputs_p1b"
os.makedirs(SAVE_DIR_P1B, exist_ok=True)

DROPOUT_RATES = {"baseline": 0.0, "dropout_0.3": 0.3, "dropout_0.5": 0.5}

histories_p1b    = {}
test_results_p1b = {}
models_p1b       = {}

for variant_name, p in DROPOUT_RATES.items():
    print(f"\n{'#'*60}")
    print(f"  Variant : {variant_name}  (dropout_p = {p})")
    print(f"{'#'*60}")

    set_seed(SEED)   # identical init for every variant — fair comparison
    model_v = ModifiedAlexNet(num_classes=NUM_CLASSES, dropout_p=p)

    history_v, ckpt_v = run_training(
        model_v, train_loader, val_loader,
        label=f"p1b_{variant_name}", save_dir=SAVE_DIR_P1B
    )
    histories_p1b[variant_name] = history_v

    # Evaluate best checkpoint on test set
    model_v.load_state_dict(torch.load(ckpt_v, map_location=DEVICE))
    model_v.to(DEVICE)
    tl_v, ta_v = evaluate(model_v, test_loader, nn.CrossEntropyLoss(), DEVICE)
    test_results_p1b[variant_name] = (tl_v, ta_v)
    models_p1b[variant_name]       = model_v
    print(f"  ✅ Test accuracy ({variant_name}): {ta_v*100:.2f}%")

print("\n✅ All Part B variants trained.")



############################################################
  Variant : baseline  (dropout_p = 0.0)
############################################################

Training : p1b_baseline
Device   : cuda  |  Epochs: 30  |  LR: 0.001  |  BS: 128
───────────────────────────────────────────────────────────────────────────
Epoch [  1/30]  Train Loss: 2.0184  Acc: 32.79%  |  Val Loss: 1.5240  Acc: 44.08%  |  LR: 0.00100  (119.1s)
Epoch [  2/30]  Train Loss: 1.5264  Acc: 44.40%  |  Val Loss: 1.3963  Acc: 48.72%  |  LR: 0.00100  (132.1s)
Epoch [  3/30]  Train Loss: 1.4113  Acc: 48.87%  |  Val Loss: 1.2439  Acc: 55.74%  |  LR: 0.00100  (124.8s)
Epoch [  4/30]  Train Loss: 1.3232  Acc: 52.05%  |  Val Loss: 1.1580  Acc: 58.84%  |  LR: 0.00100  (119.2s)
Epoch [  5/30]  Train Loss: 1.2336  Acc: 55.71%  |  Val Loss: 1.1039  Acc: 60.66%  |  LR: 0.00100  (123.7s)
Epoch [  6/30]  Train Loss: 1.1662  Acc: 58.21%  |  Val Loss: 1.0654  Acc: 62.02%  |  LR: 0.00100  (122.9s)
Epoch [  7/30]  Train Loss: 1.1

## Cell 7 — Part B: Comparison Plots & Summary Table

In [7]:
styles = {
    "baseline":    ("tab:blue",   "-",  "Baseline (p=0.0)"),
    "dropout_0.3": ("tab:orange", "--", "Dropout  (p=0.3)"),
    "dropout_0.5": ("tab:green",  ":",  "Dropout  (p=0.5)"),
}

# ── 2×2 combined panel ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for key, hist in histories_p1b.items():
    color, ls, lbl = styles[key]
    ep = range(1, len(hist["train_loss"]) + 1)
    axes[0,0].plot(ep, hist["train_loss"],          color=color, ls=ls, label=lbl)
    axes[0,1].plot(ep, hist["val_loss"],            color=color, ls=ls, label=lbl)
    axes[1,0].plot(ep, [a*100 for a in hist["train_acc"]], color=color, ls=ls, label=lbl)
    axes[1,1].plot(ep, [a*100 for a in hist["val_acc"]],   color=color, ls=ls, label=lbl)

for ax, title, ylabel in zip(axes.flat,
    ["Training Loss","Validation Loss","Training Accuracy (%)","Validation Accuracy (%)"],
    ["CE Loss","CE Loss","Accuracy (%)","Accuracy (%)"]):
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig.suptitle("Problem 1B — Modified AlexNet Dropout Comparison",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR_P1B, "p1b_combined_panel.png"), dpi=150)
plt.show()

# ── Overfitting gap plot ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for key, hist in histories_p1b.items():
    color, ls, lbl = styles[key]
    ep  = range(1, len(hist["train_acc"]) + 1)
    gap = [(tr - vl)*100 for tr, vl in zip(hist["train_acc"], hist["val_acc"])]
    ax.plot(ep, gap, color=color, linestyle=ls, label=lbl)
ax.axhline(0, color="black", lw=0.8, ls="--", alpha=0.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("Train Acc − Val Acc (pp)")
ax.set_title("Overfitting Gap — Modified AlexNet (Problem 1B)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR_P1B, "p1b_overfit_gap.png"), dpi=150)
plt.show()

# ── Confusion matrices ────────────────────────────────────────────────────────
for key, model_v in models_p1b.items():
    true_v, pred_v = get_all_predictions(model_v, test_loader, DEVICE)
    cm_v = confusion_matrix(true_v, pred_v, normalize="true")
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(cm_v, display_labels=CIFAR10_CLASSES).plot(
        ax=ax, cmap="Blues", colorbar=True, xticks_rotation=45)
    ax.set_title(f"Part B — {key} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR_P1B, f"p1b_{key}_confusion.png"), dpi=150)
    plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("  PROBLEM 1B — DROPOUT COMPARISON SUMMARY")
print("=" * 72)
print(f"  {'Variant':<20} {'Best Val Acc':>12} {'Test Acc':>10} {'Final Train':>12} {'Gap (pp)':>10}")
print("-" * 72)
for key, hist in histories_p1b.items():
    best_val    = max(hist["val_acc"]) * 100
    final_train = hist["train_acc"][-1] * 100
    final_val   = hist["val_acc"][-1]   * 100
    gap         = final_train - final_val
    _, ta       = test_results_p1b[key]
    print(f"  {key:<20} {best_val:>11.2f}% {ta*100:>9.2f}% {final_train:>11.2f}% {gap:>9.2f}pp")
print("=" * 72)
print("  Gap = Final Train Acc − Final Val Acc  (lower = less overfitting)")
print()

best_key = max(test_results_p1b, key=lambda k: test_results_p1b[k][1])
print(f"  Best variant by test accuracy : {best_key}")

# ── Key findings ──────────────────────────────────────────────────────────────
print()
print("  KEY FINDINGS FOR REPORT WRITE-UP")
print("  " + "-"*50)
for key, hist in histories_p1b.items():
    gap = (hist["train_acc"][-1] - hist["val_acc"][-1]) * 100
    _, ta = test_results_p1b[key]
    print(f"  {key:<20}  gap={gap:.2f}pp   test={ta*100:.2f}%")
print("=" * 72)



  PROBLEM 1B — DROPOUT COMPARISON SUMMARY
  Variant              Best Val Acc   Test Acc  Final Train   Gap (pp)
------------------------------------------------------------------------
  baseline                   73.76%     72.51%       71.81%     -1.89pp
  dropout_0.3                68.48%     67.03%       63.60%     -4.78pp
  dropout_0.5                59.50%     58.65%       52.35%     -7.15pp
  Gap = Final Train Acc − Final Val Acc  (lower = less overfitting)

  Best variant by test accuracy : baseline

  KEY FINDINGS FOR REPORT WRITE-UP
  --------------------------------------------------
  baseline              gap=-1.89pp   test=72.51%
  dropout_0.3           gap=-4.78pp   test=67.03%
  dropout_0.5           gap=-7.15pp   test=58.65%
